# PINN — Black-Scholes Equation (European Call Option)

$\tau = T - t$ (잔여만기)로 시간을 뒤집으면 terminal condition → IC로 변환됩니다.

$$\frac{\partial V}{\partial \tau} = \frac{1}{2}\sigma^2 S^2 \frac{\partial^2 V}{\partial S^2} + rS\frac{\partial V}{\partial S} - rV, \quad S \in [0, S_{\max}],\ \tau \in [0, T]$$

**IC (만기 payoff, $\tau=0$):** $V(S,0) = \max(S - K,\,0)$

**BC (하단):** $V(0,\tau) = 0$

**BC (상단):** $V(S_{\max},\tau) = V^{\text{BS}}(S_{\max},\tau)$ (정확해 대입)

**Exact solution (Black-Scholes formula):**
$$V(S,\tau) = S\,\mathcal{N}(d_1) - Ke^{-r\tau}\,\mathcal{N}(d_2)$$
$$d_1 = \frac{\ln(S/K)+(r+\tfrac{1}{2}\sigma^2)\tau}{\sigma\sqrt{\tau}}, \quad d_2 = d_1 - \sigma\sqrt{\tau}$$

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm
from pinns import PINN

torch.manual_seed(42)
np.random.seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

## 1. Problem Setup

In [ ]:
K     = 1.0   # Strike price
r     = 0.05  # Risk-free rate
SIGMA = 0.2   # Volatility
T     = 1.0   # Time to maturity
S_MAX = 3.0   # Upper truncation for S  (= 3K)


def bs_exact(S, tau):
    """Black-Scholes European call price.  tau = T - t >= 0."""
    S, tau = np.broadcast_arrays(
        np.asarray(S, dtype=float),
        np.asarray(tau, dtype=float),
    )
    S, tau = S.copy(), tau.copy()
    V = np.zeros_like(S)

    # At tau=0: payoff
    m0 = tau < 1e-8
    V[m0] = np.maximum(S[m0] - K, 0.0)

    # BS formula where S > 0 and tau > 0
    m = (~m0) & (S > 1e-10)
    if m.any():
        s, t = S[m], tau[m]
        d1 = (np.log(s / K) + (r + 0.5 * SIGMA**2) * t) / (SIGMA * np.sqrt(t))
        d2 = d1 - SIGMA * np.sqrt(t)
        V[m] = s * norm.cdf(d1) - K * np.exp(-r * t) * norm.cdf(d2)
    return V


# Sanity checks
S_chk = np.linspace(0.01, S_MAX, 500)
assert np.allclose(bs_exact(S_chk, np.zeros(500)), np.maximum(S_chk - K, 0), atol=1e-6), "IC check failed"
assert np.allclose(bs_exact(np.zeros(50), np.linspace(0.01, T, 50)), 0.0, atol=1e-6),      "BC@S=0 check failed"
print("Sanity checks passed.")

## 2. Data Generation

> IC(payoff)는 $S=K$ 근방에서 꺾임(kink)이 있어 학습이 어렵습니다.  
> 해당 구간 주변에 IC 포인트를 추가 oversampling 합니다.

In [ ]:
N_pde = 6000
N_ic  = 600
N_bc  = 600

# --- PDE collocation: (S, tau) in (0, S_MAX) x (0, T) ---
S_pde_np   = np.random.uniform(0.01, S_MAX, size=(N_pde, 1))
tau_pde_np = np.random.uniform(0.0,  T,     size=(N_pde, 1))
x_pde = torch.tensor(
    np.hstack([S_pde_np, tau_pde_np]), dtype=torch.float32, device=device
)

# --- IC: tau=0, V = max(S - K, 0) ---
# 2/3 uniform + 1/3 oversampled near the kink at S=K
S_unif  = np.random.uniform(0.01, S_MAX, size=(N_ic * 2 // 3, 1))
S_kink  = K + np.random.randn(N_ic // 3, 1) * 0.15
S_kink  = np.clip(S_kink, 0.01, S_MAX)
S_ic_np = np.vstack([S_unif, S_kink])

x_ic = torch.tensor(
    np.hstack([S_ic_np, np.zeros((len(S_ic_np), 1))]),
    dtype=torch.float32, device=device
)
u_ic = torch.tensor(
    np.maximum(S_ic_np - K, 0.0), dtype=torch.float32, device=device
)

# --- BC at S=0: V = 0 ---
tau_left_np = np.random.uniform(0.0, T, size=(N_bc // 2, 1))
x_bc_left   = torch.tensor(
    np.hstack([np.zeros((N_bc // 2, 1)), tau_left_np]),
    dtype=torch.float32, device=device
)
u_bc_left = torch.zeros(N_bc // 2, 1, device=device)

# --- BC at S=S_MAX: exact BS formula ---
tau_right_np = np.random.uniform(0.0, T, size=(N_bc // 2, 1))
S_right_np   = np.full((N_bc // 2, 1), S_MAX)
x_bc_right   = torch.tensor(
    np.hstack([S_right_np, tau_right_np]),
    dtype=torch.float32, device=device
)
u_bc_right = torch.tensor(
    bs_exact(S_right_np, tau_right_np), dtype=torch.float32, device=device
)

x_bc = torch.cat([x_bc_left, x_bc_right], dim=0)
u_bc = torch.cat([u_bc_left, u_bc_right], dim=0)

print(f"PDE : {x_pde.shape}")
print(f"IC  : {x_ic.shape}   target: {u_ic.shape}  (includes kink oversampling)")
print(f"BC  : {x_bc.shape}   target: {u_bc.shape}")

## 3. PDE Residual

Black-Scholes ($\tau = T-t$ 기준):

$$\underbrace{\frac{\partial V}{\partial \tau}}_{\text{V\_tau}} - \underbrace{\frac{1}{2}\sigma^2 S^2 \frac{\partial^2 V}{\partial S^2}}_{\text{V\_SS}} - \underbrace{rS\frac{\partial V}{\partial S}}_{\text{V\_S}} + rV = 0$$

In [ ]:
def bs_residual(model, x):
    """
    Black-Scholes PDE residual (forward in tau = T - t).
    x[:, 0] = S,  x[:, 1] = tau
    """
    V = model(x)                              # [N, 1]

    # First-order derivatives
    grads = torch.autograd.grad(
        V, x,
        grad_outputs=torch.ones_like(V),
        create_graph=True
    )[0]                                      # [N, 2]
    V_S   = grads[:, 0:1]
    V_tau = grads[:, 1:2]

    # Second-order derivative w.r.t. S
    V_SS = torch.autograd.grad(
        V_S, x,
        grad_outputs=torch.ones_like(V_S),
        create_graph=True
    )[0][:, 0:1]

    S = x[:, 0:1]
    return V_tau - 0.5 * SIGMA**2 * S**2 * V_SS - r * S * V_S + r * V

## 4. Model & Training

> BC와 IC(payoff)에 높은 가중치를 부여합니다.  
> 특히 $S=0$ 경계와 kink 근방의 IC 조건이 정확해야 합니다.

In [ ]:
model = PINN(
    layers=[2, 128, 128, 128, 128, 1],
    activation="tanh",
    residual=False,
    skip=True,
    use_fourier=False,
    use_ntk=False,
).to(device)

print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
model.fit(
    pde_fn=bs_residual,
    x_pde=x_pde,
    x_bc=x_bc,
    u_bc=u_bc,
    x_ic=x_ic,
    u_ic=u_ic,
    # x_ic_ut 없음 — BS는 1차 시간 PDE
    epochs=20000,
    lr=5e-4,
    w_pde=1.0,
    w_bc=5.0,
    w_ic=5.0,
    ntk_adaptive=False,
    log_every=500,
    save_path="./best_bs.pt",
)

## 5. Loss History

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.semilogy(model.loss_history["total"], label="Total", lw=1.5)
ax.semilogy(model.loss_history["pde"],   label="PDE",   lw=1)
ax.semilogy(model.loss_history["bc"],    label="BC",    lw=1)
ax.semilogy(model.loss_history["ic"],    label="IC (payoff)", lw=1)
ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
ax.set_title("Training Loss — Black-Scholes PINN")
ax.legend()
plt.tight_layout()
plt.show()

## 6. Prediction vs Exact — Heatmap

x축: 주가 $S$, y축: 잔여만기 $\tau$

In [ ]:
NS, NT = 200, 200
S_grid   = np.linspace(0.0, S_MAX, NS)
tau_grid = np.linspace(0.0, T,     NT)
SS, TauTau = np.meshgrid(S_grid, tau_grid)   # (NT, NS)

V_exact = bs_exact(SS, TauTau)

model.eval()
xt_test = torch.tensor(
    np.stack([SS.ravel(), TauTau.ravel()], axis=1),
    dtype=torch.float32, device=device
)
with torch.no_grad():
    V_pred = model(xt_test).cpu().numpy().reshape(NT, NS)

V_err  = np.abs(V_pred - V_exact)
vmin, vmax = V_exact.min(), V_exact.max()

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
kw = dict(origin="lower", extent=[0, S_MAX, 0, T], aspect="auto")

im0 = axes[0].imshow(V_exact, cmap="RdBu_r", vmin=vmin, vmax=vmax, **kw)
axes[0].set_title("Exact $V(S,\\tau)$"); axes[0].set_xlabel("$S$"); axes[0].set_ylabel("$\\tau$")
plt.colorbar(im0, ax=axes[0])

im1 = axes[1].imshow(V_pred,  cmap="RdBu_r", vmin=vmin, vmax=vmax, **kw)
axes[1].set_title("PINN $V(S,\\tau)$"); axes[1].set_xlabel("$S$"); axes[1].set_ylabel("$\\tau$")
plt.colorbar(im1, ax=axes[1])

im2 = axes[2].imshow(V_err, cmap="hot", **kw)
axes[2].set_title("Absolute Error"); axes[2].set_xlabel("$S$"); axes[2].set_ylabel("$\\tau$")
plt.colorbar(im2, ax=axes[2])

plt.tight_layout()
plt.show()

rel_l2 = np.linalg.norm(V_pred - V_exact) / (np.linalg.norm(V_exact) + 1e-10)
print(f"Relative L2 error: {rel_l2:.4e}")

## 7. Slice Comparison: $V$ vs $S$ at Fixed $\tau$

- $\tau=0$: payoff $\max(S-K,0)$ — kink at $S=K=1$
- $\tau > 0$: BS 곡선으로 부드럽게 변환

In [ ]:
tau_slices = [0.0, 0.25, 0.5, 0.75, 1.0]

fig, axes = plt.subplots(1, len(tau_slices), figsize=(16, 3), sharey=True)
for ax, tau_val in zip(axes, tau_slices):
    tau_idx = int(round(tau_val * (NT - 1)))
    ax.plot(S_grid, V_exact[tau_idx], "k-",  lw=2,   label="Exact")
    ax.plot(S_grid, V_pred[tau_idx],  "r--", lw=1.5, label="PINN")
    ax.axvline(K, color="gray", lw=0.8, linestyle=":")
    ax.set_title(f"$\\tau = {tau_val}$")
    ax.set_xlabel("$S$")
    ax.legend(fontsize=8)

axes[0].set_ylabel("$V$")
plt.suptitle("Option Price Slices: Exact vs PINN  ($K=1$ 점선)", y=1.02)
plt.tight_layout()
plt.show()

## 8. Greeks — Delta ($\partial V / \partial S$)

PINN의 장점: autograd로 **Delta, Gamma** 등 Greeks를 별도 수치미분 없이 바로 계산할 수 있습니다.

In [ ]:
# Exact Delta: N(d1)
def bs_delta(S, tau):
    S, tau = np.broadcast_arrays(np.asarray(S, dtype=float), np.asarray(tau, dtype=float))
    S, tau = S.copy(), tau.copy()
    delta = np.zeros_like(S)
    m = (S > 1e-10) & (tau > 1e-8)
    if m.any():
        s, t = S[m], tau[m]
        d1 = (np.log(s / K) + (r + 0.5 * SIGMA**2) * t) / (SIGMA * np.sqrt(t))
        delta[m] = norm.cdf(d1)
    return delta

# PINN Delta via autograd
tau_val = 0.5
S_plot  = np.linspace(0.01, S_MAX, 300)
xt_delta = torch.tensor(
    np.stack([S_plot, np.full_like(S_plot, tau_val)], axis=1),
    dtype=torch.float32, device=device, requires_grad=True
)
V_delta  = model(xt_delta)
grads_d  = torch.autograd.grad(V_delta, xt_delta, grad_outputs=torch.ones_like(V_delta))[0]
delta_pinn = grads_d[:, 0].detach().cpu().numpy()
delta_exact = bs_delta(S_plot, tau_val)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(S_plot, delta_exact, "k-",  lw=2,   label="Exact Delta $\\mathcal{N}(d_1)$")
ax.plot(S_plot, delta_pinn,  "r--", lw=1.5, label="PINN Delta")
ax.axvline(K, color="gray", lw=0.8, linestyle=":")
ax.set_xlabel("$S$"); ax.set_ylabel("$\\Delta = \\partial V / \\partial S$")
ax.set_title(f"Delta at $\\tau = {tau_val}$")
ax.legend()
plt.tight_layout()
plt.show()